# LeetCode 476 — Number Complement bằng Python

**Link đề:** https://leetcode.com/problems/number-complement/
**Độ khó:** Easy · **Chủ đề:** Bit Manipulation (thao tác bit), mặt nạ bit (bit mask)

```python
class Solution:
    def findComplement(self, num: int) -> int:
        ...
```

## Mục tiêu học
1. Hiểu "số bù" (complement): lật mọi bit `0 ↔ 1` trong biểu diễn nhị phân **không có số 0 đứng đầu**.
2. Nắm `bin()`, `int(s, 2)`, `str.translate`/`replace`, `int.bit_length()`, và các toán tử `^`, `<<`, `>>`, `|`, `~`.
3. Đi từ cách lật ký tự trong chuỗi → dựng kết quả từng bit → **mặt nạ toàn 1** `(1 << L) - 1` rồi XOR → mẹo "lan bit" (bit smearing).
4. Hiểu vì sao `~num` của Python **không** phải đáp án (số âm, bù hai).
5. Trace, chứng minh, phân tích Big-O; 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

**Số bù** của một số nguyên là số thu được khi lật mọi bit `0 → 1` và `1 → 0` trong biểu diễn nhị phân của nó (**không tính các số 0 vô nghĩa ở đầu**).

| `num` | nhị phân | lật bit | Output |
|---:|---|---|:---:|
| 5 | `101` | `010` | 2 |
| 1 | `1` | `0` | 0 |
| 10 | `1010` | `0101` | 5 |
| 7 | `111` | `000` | 0 |

**Constraints:** `1 <= num < 2^31`.

Ghi chú: bài này giống LeetCode 1009 (Complement of Base 10 Integer), khác ở chỗ 1009 cho phép `n = 0` (đáp án là 1).

Điểm mấu chốt nằm ở cụm "**không có số 0 đứng đầu**": nếu coi 5 là số 32 bit `000...0101` thì lật hết sẽ ra `111...1010` — một số rất lớn, không phải 2. Ta chỉ lật **đúng `L` bit có nghĩa**.

In [1]:
num = 5
print("bin(5) =", bin(num), "-> phần bit:", bin(num)[2:])
print("5 dạng 8 bit:", format(num, '08b'), "-> nếu lật cả 8 bit:", format(num ^ 0xFF, '08b'), "=", num ^ 0xFF, "(SAI)")
print("chỉ lật 3 bit có nghĩa: 101 -> 010 =", 0b010)

bin(5) = 0b101 -> phần bit: 101
5 dạng 8 bit: 00000101 -> nếu lật cả 8 bit: 11111010 = 250 (SAI)
chỉ lật 3 bit có nghĩa: 101 -> 010 = 2


Câu hỏi đặt ra trước khi code:
- Làm sao biết số có bao nhiêu bit có nghĩa?
- Phép nào "lật" một bit? Lật nhiều bit cùng lúc được không?
- Vì sao không dùng thẳng toán tử `~` (NOT) của Python?

## 2. Nền tảng Python cần biết

### 2.1. Chuyển qua lại giữa số và chuỗi nhị phân

- `bin(n)` → chuỗi `'0b...'`; `bin(n)[2:]` bỏ tiền tố.
- `int(s, 2)` đọc chuỗi `s` như số **cơ số 2** → số nguyên. Ví dụ `int('010', 2) == 2`.

In [2]:
print(bin(10), bin(10)[2:])
print(int('1010', 2), int('010', 2), int('0', 2))
print(int('ff', 16), int('17', 8))   # int(s, base) dùng được cho mọi cơ số 2..36

0b1010 1010
10 2 0
255 15


### 2.2. `int.bit_length()` — số bit có nghĩa

`n.bit_length()` = số bit cần để viết `n` (không tính số 0 đầu). `5 → 3` (`101`), `1 → 1`, `8 → 4` (`1000`). Tương đương `len(bin(n)) - 2` với `n > 0`.

In [3]:
for n in [1, 5, 7, 8, 10, 2**31 - 1]:
    print(f"{n:>10} = {n:b} -> bit_length = {n.bit_length()}")

         1 = 1 -> bit_length = 1
         5 = 101 -> bit_length = 3
         7 = 111 -> bit_length = 3
         8 = 1000 -> bit_length = 4
        10 = 1010 -> bit_length = 4
2147483647 = 1111111111111111111111111111111 -> bit_length = 31


### 2.3. Dịch trái `<<` và mặt nạ toàn 1

`1 << L` = `2^L` = số 1 theo sau bởi `L` số 0 (`1 << 3 = 0b1000 = 8`). Trừ đi 1 thì **mọi bit phía dưới thành 1**: `(1 << 3) - 1 = 0b0111 = 7`. Đây là **mặt nạ** (mask) gồm đúng `L` bit 1.

In [4]:
for L in range(1, 6):
    mask = (1 << L) - 1
    print(f"L={L}: 1 << L = {1 << L:>6b}, mask = {mask:>6b} ({mask})")

L=1: 1 << L =     10, mask =      1 (1)
L=2: 1 << L =    100, mask =     11 (3)
L=3: 1 << L =   1000, mask =    111 (7)
L=4: 1 << L =  10000, mask =   1111 (15)
L=5: 1 << L = 100000, mask =  11111 (31)


### 2.4. XOR với 1 để lật bit

Bảng XOR: `0 ^ 1 = 1`, `1 ^ 1 = 0` → XOR với 1 **lật** bit; XOR với 0 **giữ nguyên**. Vậy `num ^ mask` lật đúng các bit nằm dưới mặt nạ, giữ nguyên phần còn lại.

In [5]:
num = 0b101
mask = 0b111
print(f"{num:03b} ^ {mask:03b} = {num ^ mask:03b} ({num ^ mask})")
print(f"Lật riêng bit 1: {num:03b} ^ 010 = {num ^ 0b010:03b}")

101 ^ 111 = 010 (2)
Lật riêng bit 1: 101 ^ 010 = 111


### 2.5. Toán tử `~` (NOT) trong Python — cái bẫy

Python có số nguyên **không giới hạn bit** và biểu diễn số âm theo kiểu **bù hai** (two's complement) "vô hạn". `~x` bằng `-x - 1`. Vì vậy `~5 == -6`, **không phải** 2: nó lật cả "vô hạn" bit 0 ở đầu thành 1, tạo số âm.

In [6]:
print("~5 =", ~5, "| -5 - 1 =", -5 - 1)
print("~0 =", ~0)
# Muốn dùng ~ phải "cắt" bằng mặt nạ:
L = (5).bit_length()
print("~5 & mask =", ~5 & ((1 << L) - 1))

~5 = -6 | -5 - 1 = -6
~0 = -1
~5 & mask = 2


## 3. Cách 1 — Trực giác: lật ký tự trong chuỗi bit

Làm y như định nghĩa: viết ra chuỗi bit, đổi `'0'` thành `'1'` và ngược lại, rồi đọc lại thành số.

Dùng `str.translate` với bảng `str.maketrans('01', '10')`: thay **đồng thời** mọi `'0'` bằng `'1'` và mọi `'1'` bằng `'0'` (nếu dùng hai lần `replace` liên tiếp sẽ bị lỗi — xem mục lỗi).

In [7]:
table = str.maketrans('01', '10')
print("1010".translate(table))

def complement_string(num: int) -> int:
    bits = bin(num)[2:]
    flipped = bits.translate(str.maketrans('01', '10'))
    return int(flipped, 2)

for n in [5, 1, 10, 7]:
    print(n, "->", complement_string(n))

0101
5 -> 2
1 -> 0
10 -> 5
7 -> 0


Cách khác cho cùng ý: dựng chuỗi mới bằng `''.join('1' if ch == '0' else '0' for ch in bits)`.

Độ phức tạp: `O(L)` thời gian và `O(L)` bộ nhớ cho chuỗi, với `L` = số bit (≤ 31). Dễ hiểu, nhưng ta muốn làm bằng phép toán bit.

## 4. Cách 2 — Dựng kết quả từng bit

Đi qua từng bit của `num` từ thấp lên cao (`num & 1`, `num >>= 1`). Bit nào là `0` thì đặt bit tương ứng của `result` thành 1 bằng `result |= 1 << pos`.

`|` là OR theo bit: bit kết quả là 1 nếu **ít nhất một** bên là 1. `result |= 1 << pos` "bật" bit vị trí `pos`.

In [8]:
r = 0
r |= 1 << 1
print(f"bật bit 1: {r:04b}")
r |= 1 << 3
print(f"bật bit 3: {r:04b}")

def complement_bit_by_bit(num: int) -> int:
    result = 0
    pos = 0
    while num:
        if (num & 1) == 0:
            result |= 1 << pos
        num >>= 1
        pos += 1
    return result

print([complement_bit_by_bit(n) for n in [5, 1, 10, 7]])

bật bit 1: 0010
bật bit 3: 1010
[2, 0, 5, 0]


Vòng lặp tự dừng khi đã đi hết bit cao nhất (`num` thành 0) — đúng tinh thần "không tính số 0 đầu".

## 5. Bước đột phá — một phép XOR với mặt nạ

Thay vì lật từng bit, ta lật **tất cả `L` bit cùng lúc**:

> `mask = (1 << num.bit_length()) - 1` (gồm `L` bit 1) → `answer = num ^ mask`.

Ví dụ `num = 10 = 1010`, `L = 4`, `mask = 1111`, `1010 ^ 1111 = 0101 = 5`.

Một cách nhìn khác: `mask - num` cũng cho cùng kết quả (vì `num` là "tập con bit" của `mask`, trừ không có mượn).

In [9]:
num = 10
L = num.bit_length()
mask = (1 << L) - 1
print(f"num  = {num:04b}\nmask = {mask:04b}\nXOR  = {num ^ mask:04b} = {num ^ mask}")
print("mask - num =", mask - num)

num  = 1010
mask = 1111
XOR  = 0101 = 5
mask - num = 5


## 6. Lời giải chính — mổ xẻ từng đoạn rồi ghép

1. `L = num.bit_length()` — đếm số bit có nghĩa.
2. `mask = (1 << L) - 1` — tạo `L` bit 1.
3. `return num ^ mask` — lật đúng `L` bit đó.

In [10]:
def complement_mask(num: int) -> int:
    L = num.bit_length()
    mask = (1 << L) - 1
    return num ^ mask

for n in [5, 1, 10, 7, 2**31 - 1, 2**30]:
    print(f"{n:>10} ({n:b}) -> {complement_mask(n)}")

         5 (101) -> 2
         1 (1) -> 0
        10 (1010) -> 5
         7 (111) -> 0
2147483647 (1111111111111111111111111111111) -> 0
1073741824 (1000000000000000000000000000000) -> 1073741823


## 7. TRACE từng bước

### 7.1. Trace cách dựng từng bit với `num = 10` (`1010`)

| vòng | `num` (bin) | bit cuối | `pos` | hành động | `result` (bin) |
|---|---|---|---|---|---|
| 1 | `1010` | 0 | 0 | bật bit 0 | `0001` |
| 2 | `101` | 1 | 1 | bỏ qua | `0001` |
| 3 | `10` | 0 | 2 | bật bit 2 | `0101` |
| 4 | `1` | 1 | 3 | bỏ qua | `0101` |
| — | `0` | dừng | | | **5** |

In [11]:
def trace_bit_by_bit(num):
    result, pos = 0, 0
    step = 1
    while num:
        bit = num & 1
        action = "bỏ qua" if bit else f"bật bit {pos}"
        if bit == 0:
            result |= 1 << pos
        print(f"vòng {step}: num={num:>6b}, bit cuối={bit}, pos={pos}, {action:<10} result={result:b}")
        num >>= 1
        pos += 1
        step += 1
    print("Kết quả:", result)
    return result

trace_bit_by_bit(10)

vòng 1: num=  1010, bit cuối=0, pos=0, bật bit 0  result=1
vòng 2: num=   101, bit cuối=1, pos=1, bỏ qua     result=1
vòng 3: num=    10, bit cuối=0, pos=2, bật bit 2  result=101
vòng 4: num=     1, bit cuối=1, pos=3, bỏ qua     result=101
Kết quả: 5


5

### 7.2. Trace cách mặt nạ trên vài số

In [12]:
def trace_mask(num):
    L = num.bit_length()
    mask = (1 << L) - 1
    print(f"num={num:>4} = {num:0{L}b} | L={L} | mask={mask:0{L}b} | num^mask={num ^ mask:0{L}b} = {num ^ mask}")

for n in [5, 1, 10, 7, 8]:
    trace_mask(n)

num=   5 = 101 | L=3 | mask=111 | num^mask=010 = 2
num=   1 = 1 | L=1 | mask=1 | num^mask=0 = 0
num=  10 = 1010 | L=4 | mask=1111 | num^mask=0101 = 5
num=   7 = 111 | L=3 | mask=111 | num^mask=000 = 0
num=   8 = 1000 | L=4 | mask=1111 | num^mask=0111 = 7


Giải thích `{num:0{L}b}`: f-string cho phép lồng biến vào phần định dạng — `0{L}b` với `L = 4` trở thành `04b` (nhị phân, đệm đủ 4 ký tự).

## 8. Cách 4 (sáng tạo) — mặt nạ không dùng `bit_length`

### 8.1. Nâng dần mặt nạ
Bắt đầu `mask = 1`; trong khi `mask < num`, dịch trái và thêm 1: `mask = (mask << 1) | 1` → `1, 11, 111, ...` cho tới khi phủ hết `num`.

In [13]:
def complement_grow_mask(num: int) -> int:
    mask = 1
    while mask < num:
        mask = (mask << 1) | 1
    return num ^ mask

print([complement_grow_mask(n) for n in [5, 1, 10, 7, 8]])

[2, 0, 5, 0, 7]


### 8.2. "Lan bit" (bit smearing) — kiểu C/Java với số 32 bit

Lấy `m = num`, rồi `m |= m >> 1; m |= m >> 2; m |= m >> 4; m |= m >> 8; m |= m >> 16`. Sau 5 bước, mọi bit **dưới** bit 1 cao nhất đều thành 1 → `m` chính là mặt nạ. Chỉ 5 phép, không vòng lặp phụ thuộc số bit — mẹo kinh điển khi tìm lũy thừa 2 kế tiếp.

In [14]:
def complement_smear(num: int) -> int:
    m = num
    for shift in (1, 2, 4, 8, 16):
        m |= m >> shift
    return num ^ m

n = 0b1000000
m = n
for shift in (1, 2, 4, 8, 16):
    m |= m >> shift
    print(f"sau |= >> {shift:>2}: {m:07b}")
print([complement_smear(x) for x in [5, 1, 10, 7, 8]])

sau |= >>  1: 1100000
sau |= >>  2: 1111000
sau |= >>  4: 1111111
sau |= >>  8: 1111111
sau |= >> 16: 1111111
[2, 0, 5, 0, 7]


## 9. Code nộp LeetCode (Python 3)

In [15]:
class Solution:
    def findComplement(self, num: int) -> int:
        mask = (1 << num.bit_length()) - 1
        return num ^ mask

sol = Solution()
print(sol.findComplement(5))   # 2
print(sol.findComplement(1))   # 0

2
0


## 10. Vì sao đúng?

- Theo định nghĩa, đáp án chỉ liên quan tới `L = num.bit_length()` bit thấp; mọi bit từ vị trí `L` trở lên của đáp án là 0.
- `mask` có bit `i` bằng 1 ⇔ `i < L`. Với `i < L`: bit `i` của `num ^ mask` = bit `i` của `num` XOR 1 = **bit lật**. Với `i >= L`: bit `num` là 0, bit `mask` là 0 → kết quả 0.
- Vậy `num ^ mask` khớp định nghĩa ở **mọi** vị trí bit.

**Cách dựng từng bit** — invariant: sau `pos` vòng, `result` bằng phần bù của `pos` bit thấp nhất của `num` gốc, và `num` hiện tại là phần bit còn lại chưa xét. Vòng dừng đúng khi hết bit có nghĩa.

## 11. Phân tích Time / Space

Gọi `L` = số bit của `num` (≤ 31).

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Lật ký tự chuỗi | `O(L)` | `O(L)` | trực giác |
| 2. Dựng từng bit | `O(L)` | `O(1)` | luyện `&`, `>>`, `|=` |
| 3. `bit_length` + XOR | `O(1)`* | `O(1)` | **gọn nhất** |
| 4a. Nâng dần mặt nạ | `O(L)` | `O(1)` | không cần hàm có sẵn |
| 4b. Lan bit | `O(log W)` = 5 bước | `O(1)` | `W = 32` bit |

\* `bit_length` và XOR trên số nhỏ là phép hằng số trong thực tế.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", 5, 2),
    ("đề 2", 1, 0),
    ("toàn bit 1", 7, 0),
    ("lũy thừa 2", 8, 7),
    ("1010", 10, 5),
    ("2", 2, 1),
    ("max 31 bit", 2**31 - 1, 0),
    ("2^30", 2**30, 2**30 - 1),
]
sol = Solution()
for name, n, expected in cases:
    check(name, sol.findComplement(n), expected)

PASS  đề 1: 2
PASS  đề 2: 0
PASS  toàn bit 1: 0
PASS  lũy thừa 2: 7
PASS  1010: 5
PASS  2: 1
PASS  max 31 bit: 0
PASS  2^30: 1073741823


In [17]:
import random
random.seed(476)
methods = [complement_string, complement_bit_by_bit, complement_mask, complement_grow_mask, complement_smear]
for _ in range(3000):
    n = random.randint(1, 2**31 - 1)
    results = {f(n) for f in methods}
    assert len(results) == 1, (n, results)
    ans = results.pop()
    # tính chất: n + bù(n) = mask ; n & bù(n) = 0 ; bù(n) < n
    L = n.bit_length()
    assert n + ans == (1 << L) - 1 and (n & ans) == 0 and ans < n
print("3000 số ngẫu nhiên: 5 cách khớp; n + bù(n) = 2^L - 1 và n & bù(n) = 0")

3000 số ngẫu nhiên: 5 cách khớp; n + bù(n) = 2^L - 1 và n & bù(n) = 0


Hai tính chất kiểm thêm: `n` và phần bù của nó **không chung bit 1 nào** (`n & ans == 0`) và cộng lại thành mặt nạ toàn 1 — một "oracle" độc lập với cách cài đặt.

## 13. Lỗi người mới hay mắc

### Lỗi 1: dùng `~num` trực tiếp

In [18]:
def wrong_tilde(num):
    return ~num          # SAI: ra số âm

print("sai :", wrong_tilde(5))
print("đúng:", complement_mask(5))

sai : -6
đúng: 2


### Lỗi 2: lật theo 32 bit cố định
Kiểu `num ^ 0xFFFFFFFF` lật cả các số 0 đứng đầu → kết quả khổng lồ.

In [19]:
print("sai :", 5 ^ 0xFFFFFFFF)
print("đúng:", 5 ^ 0b111)

sai : 4294967290
đúng: 2


### Lỗi 3: hai lần `replace` liên tiếp
`bits.replace('0', '1').replace('1', '0')` — lần thứ nhất biến mọi `'0'` thành `'1'`, lần thứ hai biến **tất cả** (cả cũ lẫn mới) thành `'0'` → luôn ra 0.

In [20]:
bits = "1010"
print("sai :", bits.replace('0', '1').replace('1', '0'))
print("đúng:", bits.translate(str.maketrans('01', '10')))
# Cách sửa khác dùng ký tự tạm:
print("đúng (ký tự tạm):", bits.replace('0', 'x').replace('1', '0').replace('x', '1'))

sai : 0000
đúng: 0101
đúng (ký tự tạm): 0101


### Lỗi 4: mặt nạ lệch 1 bit — `1 << (L - 1)` hoặc quên `- 1`
- `(1 << L)` (quên `- 1`) chỉ có **một** bit 1 ở vị trí `L` → XOR **bật thêm** bit đó thay vì lật.
- `(1 << (L - 1)) - 1` thiếu một bit → bit cao nhất không được lật (vẫn là 1).

In [21]:
num = 5; L = num.bit_length()
print("quên -1      :", num ^ (1 << L))
print("thiếu 1 bit  :", num ^ ((1 << (L - 1)) - 1))
print("đúng         :", num ^ ((1 << L) - 1))

quên -1      : 13
thiếu 1 bit  : 6
đúng         : 2


### Lỗi 5: quên ngoặc khi viết `1 << L - 1`
Trong Python, `-` ưu tiên **cao hơn** `<<`, nên `1 << L - 1` là `1 << (L - 1)`, **không phải** `(1 << L) - 1`. Luôn thêm ngoặc.

In [22]:
L = 3
print("1 << L - 1   =", 1 << L - 1)
print("(1 << L) - 1 =", (1 << L) - 1)

1 << L - 1   = 4
(1 << L) - 1 = 7


## 14. Mẹo nhận dạng pattern

- "Lật / đảo / bù **một nhóm bit**" → **XOR với mặt nạ** có bit 1 đúng ở các vị trí cần lật.
- "Số bit có nghĩa" → `bit_length()`; "mặt nạ `k` bit 1" → `(1 << k) - 1`.
- "Bật bit `i`" → `x | (1 << i)`; "tắt bit `i`" → `x & ~(1 << i)`; "kiểm tra bit `i`" → `(x >> i) & 1`.
- Nhớ: `~` trong Python cho số âm; muốn kết quả không âm phải AND với mặt nạ.

Bài cùng họ: 1009 Complement of Base 10 Integer, 190 Reverse Bits, 191 Number of 1 Bits, 461 Hamming Distance, 231 Power of Two, 693 Binary Number with Alternating Bits.

Cell dưới đây là "bảng tra nhanh" bốn thao tác bit cơ bản trên cùng một số, để bạn thấy mỗi thao tác chỉ chạm đúng một vị trí:

In [23]:
x = 0b10110          # 22
i = 2
print(f"x               = {x:05b}")
print(f"kiểm tra bit {i}  = {(x >> i) & 1}")
print(f"bật bit 0       = {x | (1 << 0):05b}")
print(f"tắt bit {i}       = {x & ~(1 << i):05b}   (~ ở đây an toàn vì AND với x không âm)")
print(f"lật bit {i}       = {x ^ (1 << i):05b}")
print(f"lật 3 bit thấp  = {x ^ 0b111:05b}")
print("lật hai lần về như cũ:", (x ^ 0b111) ^ 0b111 == x)

x               = 10110
kiểm tra bit 2  = 1
bật bit 0       = 10111
tắt bit 2       = 10010   (~ ở đây an toàn vì AND với x không âm)
lật bit 2       = 10010
lật 3 bit thấp  = 10001
lật hai lần về như cũ: True


Nhận xét: phép **lật** (`^`) là phép duy nhất trong bốn phép "đảo ngược được" — làm hai lần thì về như cũ (`x ^ m ^ m == x`). Đây cũng là lý do XOR với mặt nạ là công cụ tự nhiên cho bài "số bù".

## 15. Biến thể

### 15.1. LeetCode 1009 — cho phép `n = 0`
`(0).bit_length() == 0` → mặt nạ `0` → kết quả `0`, nhưng đáp án đúng là `1` (bit `0` lật thành `1`). Cần xử lý riêng.

In [24]:
def bitwise_complement_1009(n: int) -> int:
    if n == 0:
        return 1
    return n ^ ((1 << n.bit_length()) - 1)

print(bitwise_complement_1009(0), bitwise_complement_1009(5), bitwise_complement_1009(10))

1 2 5


### 15.2. Lật bit trong đoạn `[i, j]` (vị trí tính từ 0, bên phải)
Mặt nạ cho đoạn: `((1 << (j - i + 1)) - 1) << i`.

In [25]:
def flip_range(x, i, j):
    mask = ((1 << (j - i + 1)) - 1) << i
    return x ^ mask

x = 0b101010
print(f"{x:06b} -> lật bit 1..3 -> {flip_range(x, 1, 3):06b}")

101010 -> lật bit 1..3 -> 100100


## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Dự đoán output
Không chạy code: `(1 << 5) - 1`, `(12).bit_length()`, và phần bù của `12` là bao nhiêu?

### Bài 2 — Hoàn thiện cách mặt nạ

In [26]:
def ex2_complement(num: int) -> int:
    # TODO 1: L = số bit có nghĩa
    # TODO 2: mask = (1 << L) - 1
    # TODO 3: trả num ^ mask
    pass

# assert ex2_complement(5) == 2
# assert ex2_complement(1) == 0

### Bài 3 — Phần bù **không dùng** `bit_length` và không dùng chuỗi

In [27]:
def ex3_complement_no_builtin(num: int) -> int:
    # TODO: tự xây mặt nạ bằng vòng lặp
    pass

# assert ex3_complement_no_builtin(10) == 5
# assert ex3_complement_no_builtin(8) == 7

### Bài 4 — Binary Number with Alternating Bits (LeetCode 693)
Trả `True` nếu các bit kề nhau của `n` luôn khác nhau. Ví dụ `5 = 101` → `True`, `7 = 111` → `False`, `10 = 1010` → `True`. Gợi ý dùng XOR với `n >> 1`.

In [28]:
def ex4_has_alternating_bits(n: int) -> bool:
    # TODO
    pass

# assert ex4_has_alternating_bits(5) is True
# assert ex4_has_alternating_bits(7) is False
# assert ex4_has_alternating_bits(10) is True
# assert ex4_has_alternating_bits(11) is False

### Bài 5 — Mini challenge: lũy thừa 2 nhỏ nhất `>= n` bằng mẹo lan bit
Ví dụ `n = 5 → 8`, `n = 8 → 8`, `n = 1 → 1`, `n = 17 → 32` (giả sử `1 <= n <= 2^30`).

In [29]:
def ex5_next_power_of_two(n: int) -> int:
    # TODO: lan bit trên (n - 1) rồi cộng 1
    pass

# assert ex5_next_power_of_two(5) == 8
# assert ex5_next_power_of_two(8) == 8
# assert ex5_next_power_of_two(1) == 1
# assert ex5_next_power_of_two(17) == 32

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: `1 << 5 = 32`. Mức 2: `12 = 1100`. Mức 3: mặt nạ `1111`, `1100 ^ 1111 = 0011`.

**Bài 2** — Mức 1: `num.bit_length()`. Mức 2: nhớ ngoặc `(1 << L) - 1`. Mức 3: `return num ^ mask`.

**Bài 3** — Mức 1: mặt nạ cần phủ `num`. Mức 2: `while mask < num`. Mức 3: `mask = (mask << 1) | 1`.

**Bài 4** — Mức 1: nếu bit xen kẽ thì `n ^ (n >> 1)` toàn bit 1. Mức 2: số toàn bit 1 `x` thỏa `x & (x + 1) == 0`. Mức 3: `x = n ^ (n >> 1); return x & (x + 1) == 0`.

**Bài 5** — Mức 1: lan bit biến `m` thành `2^k - 1`. Mức 2: làm trên `n - 1` để lũy thừa 2 không bị nhảy lên gấp đôi. Mức 3: `m = n - 1`, lan bit, `return m + 1`.

## 17. Đáp án tham khảo

**Bài 1:** `(1 << 5) - 1 = 31`, `(12).bit_length() = 4`, phần bù của 12 là `3`.

In [30]:
def ans2_complement(num):
    return num ^ ((1 << num.bit_length()) - 1)

def ans3_complement_no_builtin(num):
    mask = 1
    while mask < num:
        mask = (mask << 1) | 1
    return num ^ mask

def ans4_has_alternating_bits(n):
    x = n ^ (n >> 1)
    return (x & (x + 1)) == 0

def ans5_next_power_of_two(n):
    m = n - 1
    for shift in (1, 2, 4, 8, 16):
        m |= m >> shift
    return m + 1

assert (1 << 5) - 1 == 31 and (12).bit_length() == 4 and ans2_complement(12) == 3   # bài 1
assert ans2_complement(5) == 2 and ans2_complement(1) == 0
assert ans3_complement_no_builtin(10) == 5 and ans3_complement_no_builtin(8) == 7
assert ans4_has_alternating_bits(5) is True
assert ans4_has_alternating_bits(7) is False
assert ans4_has_alternating_bits(10) is True
assert ans4_has_alternating_bits(11) is False
assert [ans5_next_power_of_two(n) for n in [5, 8, 1, 17]] == [8, 8, 1, 32]
print("Đáp án bài 1–5: PASS")

Đáp án bài 1–5: PASS


Giải thích bài 4: với `n = 10 = 1010`, `n >> 1 = 0101`, XOR = `1111`; `1111 & 10000 = 0` → toàn bit 1 → xen kẽ. Với `n = 11 = 1011`, XOR với `0101` = `1110`, `1110 & 1111 = 1110 ≠ 0` → không xen kẽ.

## 18. Template nhớ nhanh + checklist submit

```python
mask = (1 << num.bit_length()) - 1   # L bit 1
return num ^ mask                    # lật đúng L bit
```

**Checklist trước khi submit**
- [ ] Không dùng `~num` trần (ra số âm)?
- [ ] Mặt nạ có **đúng** `L` bit (có ngoặc, có `- 1`)?
- [ ] Không lật theo 32 bit cố định?
- [ ] Ca `num = 1` → `0`; ca toàn bit 1 (`7`) → `0`; lũy thừa 2 (`8`) → `7`?
- [ ] Nếu là bài 1009, đã xử lý `n = 0` → `1`?